Go include nella libreria standard il package `net/rpc`, che permette a un client di invocare metodi eseguiti in un altro processo anche remoto.  

L'idea è la stessa vista con la calcolatrice in RPyC: il server implementa le operazioni e il client ne richiede l'esecuzione.  
Per invocare il metodo, in Go il client lo indica con una stringa come `"NomeDelServizio.NomeDelMetodo"` (es. `"Arithmetic.Multiply"`) e passa gli argomenti e un puntatore alla variabile che riceverà il risultato.  
La libreria si occupa di codificare i dati, inviare la richiesta al server, individuare il metodo corretto e decodificare la risposta.

Il meccanismo è **type-safe**, nel senso che i metodi del server hanno firme Go tipizzate e i dati ricevuti devono essere compatibili con i tipi previsti. Il nome del metodo remoto resta però una stringa: se ad es. il client scrive per sbaglio `"Arithmetic.Mutiply"` l'errore emerge durante la chiamata, non in compilazione.

Si distinguono in Go due modalità di connessione: **TCP diretto** e **connessione RPC avviata tramite HTTP**.

Per definire un servizio, ciò che si fa è creare un tipo (tipicamente una `struct`) e gli si associano i metodi da pubblicare:
```go
func (s *Servizio) Metodo(input TipoInput, reply *TipoRisposta) error {
    // implementazione
}
```
`s *Servizio` è il **receiver** (l'istanza del servizio dove è eseguito il metodo), `input` è il primo argomento (può essere una `struct` che raggruppa più dati, ad es. due operandi della calcolatrice oppure un puntatore a quella strcut), `reply` è il secondo argomento **e deve necessariamente essere un puntatore** (il metodo scrive il risultato nella variabile indicata dal puntatore) e `error` è l'unico valore restituito direttamente dal metodo: se l'operazione riesce vale `nil`.

Il tipo di servizio e i metodi devono essere **esportati** (cioè iniziare con lettera maiuscola), così come i tipi dei parametri e del risultato.

---
Vediamo un esempio pratico su come definire un servizio che poi sarà chiamato via RPC:
```go 
package arith
import "errors"

type Arith struct{}
type Args struct {
    A, B int
}
type Result int
type Quotient struct {
    Quo, Rem int //risultato della divisione: quoziente e resto
}

func (t *Arith) Multiply(args *Args, res *Result) error {
    *res = Result(args.A * args.B)
    return nil
}
func (t *Arith) Divide(args *Args, quo *Quotient) error {
    if args.B == 0 {
        return errors.New("division by zero")
    }
    quo.Quo = args.A / args.B
    quo.Rem = args.A % args.B
    return nil
}
```
Il package `arith` è il package della calcolatrice (un package in Go è un gruppo di file che raccolgono codice collegato). Scrivere all'inizio `package arith` significa che il file appartiene ad esso.  

`Arith` è il tipo del servizio su cui sono definiti i metodi `Multiply` e `Divide`, è una struct vuota in questo caso perché queste operazioni dipendono solo dagli argomenti ricevuti e non hanno bisogno di memorizzare dati interni tra una chiamata e l'altra.

`Args` è struct con due operandi interni per poterli passare come argomento ai metodi, `Result` è un tipo `int` per il risultato del prodotto e `Quotient` è una struct con due campi interi per il quoziente e il resto della divisione.

`Multiply` riceve `args *Args` e `res *Result`, calcola `args.A * args.B` e lo scrive in `res` tramite il puntatore. Il metodo restituisce poi `nil` per segnalare il successo.

`Divide` controlla prima il divisore: se `args.B` è zero restituisce un errore, altrimenti scrive il quoziente in `quo.Quo` e il resto in `quo.Rem` e restituisce `nil`.

### Lato Server
Il server deve pubblicare i metodi della calcolatrice e mettersi in ascolto sulla rete, per farlo due operazioni separate:
1. **Crea e registra il servizio**: crea un'istanza di `Arith` e la consegna a `net/rpc` tramite `rpc.Register()`. Il `.Register()` userebbe il nome del tipo.  
Si usa invece `rpc.RegisterName("NomeServizio", &Arith{})` se si vuole dare un nome diverso al servizio --> in questo modo il client userà `"Arithmetic.Multiply"` invece di `"Arith.Multiply"`.
1. **Apre il listener**: `net.Listen` apre la porta TCP `12345` e restituisce un *listener*, l'oggetto attraverso cui il programma può accettare le connessioni
2. **Accetta le connessioni**: `server.Accept(listener)` entra in un ciclo infinito e accetta le connessioni in arrivo.  
`net/rpc` gestisce le connessioni e le richieste via goroutine di modo che il server possa servire più chiamate contemporaneamente.  
Si parla di concorrenza *out of the box* perché il programmatore non deve preoccuparsi di creare le goroutine, la libreria lo fa automaticamente, però se i metodi modificano uno stato condiviso sta allo sviluppatore sincronizzare quegli accessi.

Quando arriva una richiesta per `"Arithmetic.Multiply"` la libreria trova il servizio registrato come `Arithmetic`, decodifica gli argomenti, esegue `Multiply` e rimanda al client il risultato o l'errore.

```go 
package main

import (
	"log"
	"net"
	"net/rpc"

	"gorpc_arith/arith"
)

func main() {
	arithService := new(arith.Arith)
	server := rpc.NewServer()

	if err := server.RegisterName("Arithmetic", arithService); err != nil {
		log.Fatal("Registrazione del servizio: ", err)
	}

	lis, err := net.Listen("tcp", ":12345")
	if err != nil {
		log.Fatal("Apertura del listener: ", err)
	}
	defer lis.Close()

	log.Printf("Server RPC in ascolto su %s", lis.Addr())
	server.Accept(lis)
}
```

### Lato Client
Il client apre una connessione con `rpc.Dial("tcp", "host:porta")` indicando dove si trova il server, oppure con `rpc.DialHTTP` per la variante HTTP. Una volta connesso indica **quale metodo vuole eseguire** (es. `"Arithmetic.Multiply"`) e passa gli argomenti e un puntatore alla variabile che riceverà il risultato.  

Per eseguire i metodi remoti si può usare `client.Call` (sincrono) oppure `client.Go` (asincrono):
- `client.Call("Arithmetic.Multiply", args, &result)` blocca il client finché non riceve la risposta dal server. `Call` restituisce l'eventuale errore; in caso di successo la libreria avrà già scritto il risultato in `result` tramite il puntatore.
- `client.Go("Arithmetic.Multiply", args, &result, nil)` restituisce subito una struct `*rpc.Call` senza aspettare la risposta remota. Il quarto argomento `nil` chiede alla libreria di creare un canale `Done` automaticamente. `Done` è un canale che segnala quando la chiamata è terminata, a quel punto il client può controllare `Error` e se è tutto ok leggere il risultato in `result`.  

```go
package main

import (
	"fmt"
	"log"
	"net/rpc"
	"os"
	"strconv"

	"gorpc_arith/arith"
)

func main() {
	if len(os.Args) < 3 {
		log.Fatal("Uso: client_tcp <num1> <num2>")
	}

	n1, err := strconv.Atoi(os.Args[1])
	if err != nil {
		log.Fatal("Primo numero non valido: ", err)
	}

	n2, err := strconv.Atoi(os.Args[2])
	if err != nil {
		log.Fatal("Secondo numero non valido: ", err)
	}

	client, err := rpc.Dial("tcp", "localhost:12345")
	if err != nil {
		log.Fatal("Connessione al server: ", err)
	}
	defer client.Close()

	args := arith.Args{A: n1, B: n2}

	// Slide 107: moltiplicazione sincrona.
	var mulReply arith.Result
	if err := client.Call("Arithmetic.Multiply", args, &mulReply); err != nil {
		log.Fatal("Moltiplicazione: ", err)
	}
	fmt.Printf("%d * %d = %d\n", args.A, args.B, mulReply)

	// Slide 108: divisione asincrona.
	divReply := new(arith.Quotient)
	divCall := client.Go("Arithmetic.Divide", args, divReply, nil)

	//qui si potrebbe eseguire altro codice vista la chiamata asincrona

	<-divCall.Done
	if divCall.Error != nil {
		log.Fatal("Divisione: ", divCall.Error)
	}
	fmt.Printf("%d / %d = %d, resto %d\n",
		args.A, args.B, divReply.Quo, divReply.Rem)
}
```
Qui il client legge due numeri dalla riga di comando tramite `os.Args` (che contiene gli argomenti con cui si è avviato il programma: scrivendo `go run client_tcp.go 7 3` allora `os.Args[1]` è `"7"` e `os.Args[2]` è `"3"`) e li mette in `args`.  
Li usa poi nella chiamata sincrona per moltiplicazione e in quella asincrona per la divisione.

---

`net/rpc` lascia all'applicazione la politica da seguire in caso di guasto --> la libreria non imposta una scadenza per ogni RPC, non fa retry e non elimina duplicati.

Per questo motivo si parla di semantica **best-effort** riguardo Go RPC: il client non ha alcuna garanzia se non riceve una risposta; la procedura potrebbe non essere mai arrivata al server oppure il server potrebbe averla eseguita.... sta quindi allo sviluppatore integrare eventualmente semantiche migliori.

Per quel che riguarda **marshalling** e **unmarshalling**, `net/rpc` svolge queste operazioni **automaticamente** usando il package `encoding/gob` (dove `gob` sta per *Go binary*): i dati vengono convertiti in un formato binario e trasmessi al server, che li ricostruisce. Chiaramente i puntatori non vengono trasmessi, ma i dati a cui puntano.  
Si usa l'encoding `gob` esplicitamente solo se si vogliono codificare dati fuori da una RPC, ad es. per salvarli su file.

Riassumendo le caratteristiche principali di Go RPC:
- **Accesso**: la libreria `net/rpc` nasconde marshalling/unmarshalling, che tra l'altro sono gestiti automaticamente. Il client riconosce comunque la chiamata remota dalla sintassi esplicita `client.Call` e `client.Go`.
- **Locazione**: il client deve conoscere un endpoint (i.e. host e porta), infatti la registrazione interna individua i metodi solo dopo che la connessione è stata stabilita.
- **Concorrenza**: il server può servire più connessioni e richieste concorrenti: la sincronizzazione dei dati condivisi resta a carico dello sviluppatore.
- **Fallimenti**: semantica best effort --> timeout, retry e deduplicazione richiedono implementazione aggiuntiva.
- **Replicazione**: come visto per tutti i sistemi RPC finora, sta all'applicazione/infrastruttura organizzare la distribuzione delle richieste tra le repliche, magari tramite un load balancer. 

Vediamo ora un sunto di confronto tra Sun RPC, Java RMI, RPyC e Go RPC in base a 5 tipi di trasparenza:

**1. Trasparenza di accesso**: quanto assomiglia una chiamata remota a una locale?
- **Sun RPC**: qui la trasparenza è parziale: gli stub generati a partire dall'IDL nasconono lo scambio dei messaggi, ma il codice deve seguire le convenzioni di `rpcgen` (numero di versione, nomi generati e suffisso `_svc` lato server)
- **Java RMI**: il client invoca un metodo su un riferimento remoto con una sintassi simile a quella di una chiamata locale. Deve però gestire esplicitamente `RemoteException`
- **RPyC**: il client usa un proxy per accedere ai metodi e agli oggetti remoti con la normale sintassi python, ad es. `conn.root.add(...)`
- **Go RPC**: la libreria gestisce lo scambio dei dati; buon livello di trasparenza, anche se nel codice la chiamata remota è esplicita tramite `client.Call` o `client.Go`. L'iniziale maiuscola è condizione necessaria per esporre i metodi lato server

**2. Trasparenza di locazione**: cosa deve sapere il client per raggiungere il servizio?
- **Sun RPC**: il client deve conoscere l'host. Contatta allora il **port mapper** su una porta nota e ottiene la porta effettiva su cui risponde il servizio RPC.
- **Java RMI**: il client raggiunge l'**RMI Registry** e cerca il servizio per nome. Il registry gli restituisce allora il riferimento remoto, i.e. lo stub attraverso cui invocare metodi
- **RPyC e Go RPC**: il client deve conoscere host e porta (endpoint) del server. In Go `RegisterName` associa un nome al servizio *dentro* il server già raggiunto; il client usa quel nome per scegliere il metodo da invocare.

**3. Trasparenza della concorrenza**: chi gestisce le richieste che arrivano contemporaneamente?

Il supporto alla concorrenza è automatico in **RPyC e Go RPC** (es. il server può accettare più connessioni e gestire richieste in goroutine). Però se i metodi remoti modificano uno stato condiviso, lo sviluppatore deve proteggere l'accesso concorrente.

**4. Trasparenza dei fallimenti**: che garanzie si hanno se una richiesta/risposta vanno perse?

RPyC e Go RPC seguono semantica **best-effort** senza quindi alcuna garanzia.  
Per Sun RPC **at least once**, per Java RMI **at most once**.

**5. Trasparenza della replicazione**: il client deve sapere quante copie del server esistono?

Per tutti gli approcci esaminati non c'è questa trasparenza. Una proposta per ottenerla è aggiungere **un load balancer che faccia da proxy RPC**: il client invia la richiesta a un unico indirizzo --> il proxy sceglie una replica, le inoltra la chiamata e restituisce al client la risposta. Si possono così aggiungere/rimuovere repliche senza cambiare l'indirizzo usato dal client.